In [2]:
%load_ext autoreload 
%autoreload 2

Alter data prep to be exactly like the Deep Triangle paper. Each Dev period will have a target SEQUENCE and an input SEQUENCE.
<br><br> Later down the line we can choose it to be either an Autoregressive MODEL or a SEQ2SEQ model. 


In [22]:

from rnn_reserving.data_import import read_local_raw_data, process_data, split_data
import pandas as pd


In [3]:
df_cas = read_local_raw_data()
    
df_cas = process_data(df_cas)
df_cas = split_data(df_cas)

In [14]:
df_test = df_cas[df_cas['GRCODE'] == 43][
    ['AccidentYear',
    'DevelopmentLag',
    'incurred_loss_ratio',
    'paid_loss_ratio',
    'case_loss_ratio',
    'calendar_year',
    'GRCODE_mapped',
    'bucket'
    ]
].copy()

In [18]:
triangle = (
    df_test
    .pivot_table(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="paid_loss_ratio",
        aggfunc="sum" 
    )
    .sort_index()
)
triangle

DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


The below shows train in blue, val in green and test in red! Yay! 


In [41]:
ilr_triangle = (
    df_test
    .pivot(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="paid_loss_ratio"
    )
    .sort_index()
)

bucket_triangle = (
    df_test
    .pivot(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="bucket"
    )
    .sort_index()
)

def style_bucket(val):
    if val == "train":
        return "background-color: #cce5ff; color: #003366;"
    if val == "validation":
        return "background-color: #d4edda; color: #155724;"
    if val == "test":
        return "background-color: #f8d7da; color: #721c24;"
    return ""


styled = (
    ilr_triangle
    .style
    .apply(
        lambda _: bucket_triangle.applymap(style_bucket),
        axis=None
    )
)

styled


C:\Users\TobyCook\AppData\Local\Temp\ipykernel_6544\2490517049.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lambda _: bucket_triangle.applymap(style_bucket),


DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


In [ ]:
# We do need to handle the val data having a full history for input but only a partial for target though..

In [84]:
input_seqs_train = []
input_seqs_test = []
input_seqs_val = []
target_seqs_train = []
target_seqs_test = []
target_seqs_val = []
lengths_train = []
lengths_test = []
lengths_val = [] 
ids_test = []
ids_train = []
ids_val = []

feature_cols = ['paid_loss_ratio']

for development_lag in range(2, df_test['DevelopmentLag'].max() + 1, 1):
    
    df_filtered =  df_test[
        df_test['DevelopmentLag'] <= development_lag
    ].copy()
    
    split = df_filtered[df_filtered['DevelopmentLag'] == development_lag]['bucket'].values[0]

    for (ay, cc), group in df_filtered.groupby(['AccidentYear', 'GRCODE_mapped']):
        group = group.sort_values('DevelopmentLag')

        if len(group) < 2:
            # skip because we need at least one input and one target
            print('SKIP', ay, cc, development_lag)
            continue
        

        seqs = group[feature_cols].values

        input_seq = seqs[:-1] # hide the last one!
        target_seq = seqs[1:, 0] # single target only 

        if split == 'train':
            print(input_seq)
            input_seqs_train.append(input_seq)
            target_seqs_train.append(target_seq)
            lengths_train.append(len(input_seq))
            ids_train.append((ay, cc, development_lag, split))
        elif split == 'validation':
            print('val')
            input_seqs_val.append(input_seq)
            target_seqs_val.append(target_seq)
            lengths_val.append(len(input_seq))
            ids_val.append((ay, cc, development_lag, split))
        
        elif split == 'test':
            input_seqs_test.append(input_seq)
            target_seqs_test.append(target_seq)
            lengths_test.append(len(input_seq))
            ids_test.append((ay, cc, development_lag, split))
        else:
            raise ValueError(f"Unknown split: {split}")


[[0.14860335]]
[[0.27414147]]
[[0.34471048]]
[[0.27031697]]
[[0.27359207]]
[[0.23743526]]
[[0.29491844]]
[[0.28211804]]
[[0.26857593]]
[[0.23709133]]
[[0.14860335]
 [0.37206704]]
[[0.27414147]
 [0.51247432]]
[[0.34471048]
 [0.82594668]]
[[0.27031697]
 [0.68678503]]
[[0.27359207]
 [0.58093102]]
[[0.23743526]
 [0.56633844]]
[[0.29491844]
 [0.63189766]]
[[0.28211804]
 [0.54613786]]
[[0.26857593]
 [0.49960579]]
[[0.23709133]
 [0.47123156]]
[[0.14860335]
 [0.37206704]
 [0.48156425]]
[[0.27414147]
 [0.51247432]
 [0.69415908]]
[[0.34471048]
 [0.82594668]
 [1.16827984]]
[[0.27031697]
 [0.68678503]
 [0.9010367 ]]
[[0.27359207]
 [0.58093102]
 [0.81256556]]
[[0.23743526]
 [0.56633844]
 [0.76077181]]
[[0.29491844]
 [0.63189766]
 [0.7949067 ]]
[[0.28211804]
 [0.54613786]
 [0.665078  ]]
[[0.26857593]
 [0.49960579]
 [0.60254853]]
[[0.23709133]
 [0.47123156]
 [0.57413444]]
[[0.14860335]
 [0.37206704]
 [0.48156425]
 [0.63687151]]
[[0.27414147]
 [0.51247432]
 [0.69415908]
 [0.75697094]]
[[0.34471048]
 [

In [63]:
input_seqs_train

[array([[0.14860335]]),
 array([[0.27414147]]),
 array([[0.34471048]]),
 array([[0.27031697]]),
 array([[0.27359207]]),
 array([[0.23743526]]),
 array([[0.29491844]]),
 array([[0.28211804]]),
 array([[0.26857593]]),
 array([[0.23709133]]),
 array([[0.14860335],
        [0.37206704]]),
 array([[0.27414147],
        [0.51247432]]),
 array([[0.34471048],
        [0.82594668]]),
 array([[0.27031697],
        [0.68678503]]),
 array([[0.27359207],
        [0.58093102]]),
 array([[0.23743526],
        [0.56633844]]),
 array([[0.29491844],
        [0.63189766]]),
 array([[0.28211804],
        [0.54613786]]),
 array([[0.26857593],
        [0.49960579]]),
 array([[0.23709133],
        [0.47123156]]),
 array([[0.14860335],
        [0.37206704],
        [0.48156425]]),
 array([[0.27414147],
        [0.51247432],
        [0.69415908]]),
 array([[0.34471048],
        [0.82594668],
        [1.16827984]]),
 array([[0.27031697],
        [0.68678503],
        [0.9010367 ]]),
 array([[0.27359207],
      

In [79]:
from collections import defaultdict

feature_cols = ['paid_loss_ratio']

data = {
    'train': defaultdict(list),
    'validation': defaultdict(list),
    'test': defaultdict(list),
}

# Group once
for (ay, cc), group in df_test.groupby(['AccidentYear', 'GRCODE_mapped']):
    group = group.sort_values('DevelopmentLag')

    seqs = group[feature_cols].values
    dev_lags = group['DevelopmentLag'].values
    splits = group['bucket'].values

    # iterate over valid sequence endpoints
    for i in range(1, len(group)):
        development_lag = dev_lags[i]
        split = splits[i]

        input_seq = seqs[:i]
        target_seq = seqs[1:i+1, 0]

        data[split]['inputs'].append(input_seq)
        data[split]['targets'].append(target_seq)
        data[split]['lengths'].append(len(input_seq))
        data[split]['ids'].append((ay, cc, development_lag, split))

In [82]:
input_seqs_train

[array([[0.14860335]]),
 array([[0.27414147]]),
 array([[0.34471048]]),
 array([[0.27031697]]),
 array([[0.27359207]]),
 array([[0.23743526]]),
 array([[0.29491844]]),
 array([[0.28211804]]),
 array([[0.26857593]]),
 array([[0.23709133]]),
 array([[0.14860335],
        [0.37206704]]),
 array([[0.27414147],
        [0.51247432]]),
 array([[0.34471048],
        [0.82594668]]),
 array([[0.27031697],
        [0.68678503]]),
 array([[0.27359207],
        [0.58093102]]),
 array([[0.23743526],
        [0.56633844]]),
 array([[0.29491844],
        [0.63189766]]),
 array([[0.28211804],
        [0.54613786]]),
 array([[0.26857593],
        [0.49960579]]),
 array([[0.23709133],
        [0.47123156]]),
 array([[0.14860335],
        [0.37206704],
        [0.48156425]]),
 array([[0.27414147],
        [0.51247432],
        [0.69415908]]),
 array([[0.34471048],
        [0.82594668],
        [1.16827984]]),
 array([[0.27031697],
        [0.68678503],
        [0.9010367 ]]),
 array([[0.27359207],
      

In [80]:
sorted([len(x) for x in input_seqs_train])

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7]

In [81]:
sorted([len(x) for x in data['train']['inputs']])

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 3,
 3,
 3,
 3,
 3,
 4,
 4,
 4,
 4,
 5,
 5,
 5,
 6,
 6,
 7]

In [49]:
target_seqs

[array([0.37206704]),
 array([0.51247432]),
 array([0.82594668]),
 array([0.68678503]),
 array([0.58093102]),
 array([0.56633844]),
 array([0.63189766]),
 array([0.54613786]),
 array([0.49960579]),
 array([0.47123156]),
 array([0.37206704, 0.48156425]),
 array([0.51247432, 0.69415908]),
 array([0.82594668, 1.16827984]),
 array([0.68678503, 0.9010367 ]),
 array([0.58093102, 0.81256556]),
 array([0.56633844, 0.76077181]),
 array([0.63189766, 0.7949067 ]),
 array([0.54613786, 0.665078  ]),
 array([0.49960579, 0.60254853]),
 array([0.47123156, 0.57413444]),
 array([0.37206704, 0.48156425, 0.63687151]),
 array([0.51247432, 0.69415908, 0.75697094]),
 array([0.82594668, 1.16827984, 1.37323824]),
 array([0.68678503, 0.9010367 , 0.99237369]),
 array([0.58093102, 0.81256556, 0.91163598]),
 array([0.56633844, 0.76077181, 0.85602482]),
 array([0.63189766, 0.7949067 , 0.89984744]),
 array([0.54613786, 0.665078  , 0.73054232]),
 array([0.49960579, 0.60254853, 0.6723348 ]),
 array([0.47123156, 0.5741

In [50]:
input_seqs

[array([[0.14860335]]),
 array([[0.27414147]]),
 array([[0.34471048]]),
 array([[0.27031697]]),
 array([[0.27359207]]),
 array([[0.23743526]]),
 array([[0.29491844]]),
 array([[0.28211804]]),
 array([[0.26857593]]),
 array([[0.23709133]]),
 array([[0.14860335],
        [0.37206704]]),
 array([[0.27414147],
        [0.51247432]]),
 array([[0.34471048],
        [0.82594668]]),
 array([[0.27031697],
        [0.68678503]]),
 array([[0.27359207],
        [0.58093102]]),
 array([[0.23743526],
        [0.56633844]]),
 array([[0.29491844],
        [0.63189766]]),
 array([[0.28211804],
        [0.54613786]]),
 array([[0.26857593],
        [0.49960579]]),
 array([[0.23709133],
        [0.47123156]]),
 array([[0.14860335],
        [0.37206704],
        [0.48156425]]),
 array([[0.27414147],
        [0.51247432],
        [0.69415908]]),
 array([[0.34471048],
        [0.82594668],
        [1.16827984]]),
 array([[0.27031697],
        [0.68678503],
        [0.9010367 ]]),
 array([[0.27359207],
      

In [ ]:
lengths

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 4,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 5,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 6,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 7,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 8,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9,
 9]